In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, rand, when, hour, avg, count, window
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

In [0]:
# 1. CLOUD INITIALIZATION (Requirement: LO2)
# Configured for AWS S3 connectivity
spark = SparkSession.builder \
    .appName("Ravensbourne-BigData-Assessment") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

In [0]:
from pyspark.sql.functions import current_timestamp


BUCKET = "bda-fraud-detection"
RAW_PATH = f"s3a://bda-fraud-detection/raw/transactions.parquet"
OUTPUT_PATH = f"s3a://bda-fraud-detection/results/fraud_flags/"

# Generating 1 million rows to justify Big Data tools
print("Step 1: Generating Synthetic Big Data...")
num_rows = 1000000 
raw_data = spark.range(0, num_rows) \
    .withColumn("transaction_id", col("id")) \
    .withColumn("user_id", (rand(seed=42) * 10000).cast("int")) \
    .withColumn("amount", (rand(seed=43) * 2000).cast("double")) \
    .withColumn("timestamp", (current_timestamp().cast("long") - (rand() * 3600 * 24 * 30)).cast("timestamp")) \
    .withColumn("is_fraud", when((col("amount") > 1500) & (rand() > 0.7), 1).otherwise(0))

# STORAGE: Writing raw data to S3 (Requirement: Data Storage)
raw_data.write.mode("overwrite").parquet(RAW_PATH)

Step 1: Generating Synthetic Big Data...


In [0]:
#PROCESSING & FEATURE ENGINEERING (Requirement: Processing) ---
print("Step 2: Processing and Feature Engineering...")
df = spark.read.parquet(RAW_PATH)

# Innovation: Temporal Feature Extraction
df_processed = df.withColumn("tx_hour", hour(col("timestamp"))) \
                 .withColumn("is_high_value", when(col("amount") > 1000, 1).otherwise(0))

# Strategy: Behavioral Aggregation (Velocity check)
user_window = df_processed.groupBy("user_id").agg(avg("amount").alias("avg_spend"))
df_final = df_processed.join(user_window, "user_id")

Step 2: Processing and Feature Engineering...


In [0]:
#ANALYSIS & MACHINE LEARNING (Requirement: Analysis) ---
print("Step 3: Training Fraud Detection Model...")

# Preparing features for Spark MLlib
assembler = VectorAssembler(
    inputCols=["amount", "tx_hour", "is_high_value", "avg_spend"],
    outputCol="raw_features"
)

# Innovation: Feature Scaling for high-variance financial data
scaler = StandardScaler(inputCol="raw_features", outputCol="features")

# Using Gradient-Boosted Trees (Advanced analysis for fraud)
gbt = GBTClassifier(labelCol="is_fraud", featuresCol="features", maxIter=20)

# Implementation of a unified Pipeline Strategy
pipeline = Pipeline(stages=[assembler, scaler, gbt])

# Train/Test Split
train, test = df_final.randomSplit([0.8, 0.2], seed=42)
model = pipeline.fit(train)

# Predictions and Impact Evaluation
predictions = model.transform(test)
evaluator = BinaryClassificationEvaluator(labelCol="is_fraud", metricName="areaUnderPR")
auprc = evaluator.evaluate(predictions)

print(f"Strategic Analysis Complete. AUPRC Score: {auprc:.4f}")

Step 3: Training Fraud Detection Model...
Strategic Analysis Complete. AUPRC Score: 0.2947


In [0]:
#FINAL STORAGE ---
# Requirement: Addressing analysis output
predictions.select("transaction_id", "probability", "prediction") \
           .write.mode("overwrite").parquet(OUTPUT_PATH)

print(f"Project successfully implemented. Results saved to: {OUTPUT_PATH}")
spark.stop()

Project successfully implemented. Results saved to: s3a://bda-fraud-detection/results/fraud_flags/
